# 💰 Cost, Caching & Robustness — pi.dev

Notebook **080** in the pi.dev series. The earlier notebooks got answers back; this
one is about running LLM calls **in production**: knowing what each call *costs*,
letting the provider **cache** repeated context, and making requests **robust** —
cancellable, time-bounded, retried, and inspectable.

Four short sections, each self-contained:

1. **Cost tracking** — real per-token pricing + `calculateCost`
2. **Prompt caching** — `cacheRetention`, `sessionId`, `usage.cacheRead/Write`
3. **Cancellation** — `AbortController` + `signal`
4. **Timeouts, retries & introspection** — `timeoutMs`, `maxRetries`, `onPayload`, `onResponse`

> **Kernel:** Deno (pi.dev is ESM-only TypeScript; the Deno kernel runs it natively with top-level await).

## Setup — load your Azure keys

`loadEnvUp()` walks up the directory tree and loads every `.env` it finds, so your
`AZURE_PI_TEST_*` keys can live **outside** the git repo. Each section below registers
Azure via the shared `registerAzure()` helper.

In [ ]:
import { loadEnvUp } from "playground/env";

// Load AZURE_PI_TEST_* from any .env above this folder.
const env = await loadEnvUp();
console.log(`\n${env.loaded.length} variable(s) loaded from ${env.files.length} .env file(s).`);

## 1. Cost tracking 💵

In notebook 010 the model's `cost` was all zeros, so `usage.cost` was always `$0`.
Cost is simply **tokens × per-token price**, so if you want meaningful numbers you must
put your deployment's real pricing on the `Model`. `registerAzure({ modelOverrides })`
lets us override just the `cost` field.

Prices are **per token** — Azure/OpenAI quote per *million* tokens, so divide by 1e6.
Here we use illustrative GPT-4o-class pricing ($2.50 / 1M input, $10 / 1M output).

`calculateCost(model, usage)` is the same function pi-ai uses internally — we call it
ourselves to show the number is reproducible from `usage` + the model's price table.

In [ ]:
import { registerAzure } from "playground/azure";
import { calculateCost, type Context } from "@earendil-works/pi-ai";

// Register Azure with REAL per-token pricing so usage.cost is meaningful.
const { models, model, modelId } = registerAzure({
  modelOverrides: {
    cost: { input: 2.5 / 1_000_000, output: 10 / 1_000_000, cacheRead: 0, cacheWrite: 0 },
  },
});
console.log(`Model "${modelId}" priced at $2.50/1M in, $10/1M out.\n`);

const ctx1: Context = {
  systemPrompt: "You are concise.",
  messages: [{ role: "user", content: "In one sentence, what is a token in an LLM?", timestamp: Date.now() }],
};

const reply1 = await models.completeSimple(model, ctx1);
const text1 = reply1.content.filter((b) => b.type === "text").map((b) => (b as { text: string }).text).join("");
console.log("💬", text1);

console.log("\nusage:", JSON.stringify(reply1.usage, null, 2));

// calculateCost recomputes the cost from usage + the model's price table.
const recomputed = calculateCost(model, reply1.usage);
console.log(`\nreply.usage.cost.total : $${reply1.usage.cost.total.toFixed(8)}`);
console.log(`calculateCost(...) total: $${recomputed.total.toFixed(8)}`);
console.log(`match: ${Math.abs(recomputed.total - reply1.usage.cost.total) < 1e-12}`);

## 2. Prompt caching 🗄️

Re-sending a big, stable prefix (a long system prompt, tool definitions, a document)
on every turn is wasteful. Many providers can **cache** that prefix and bill it at a
reduced rate. pi-ai exposes this through `StreamOptions`:

- **`cacheRetention`** — `"none"` | `"short"` | `"long"` (default `"short"`). How long the
  provider should retain the cached prefix.
- **`sessionId`** — an opaque id for session-aware/cache-affinity routing, so repeat calls
  hit the same cached copy.
- **`usage.cacheRead` / `usage.cacheWrite`** — tokens served from / written to the cache.

Below we pass both options. If this Azure deployment doesn't implement prompt caching,
the cache counters stay `0` — that's expected; the point is the **API surface** you'd use
against a caching-capable model (e.g. Anthropic Claude).

In [ ]:
const ctx2: Context = {
  systemPrompt: "You are a helpful assistant.",
  messages: [{ role: "user", content: "List three primary colors.", timestamp: Date.now() }],
};

const reply2 = await models.completeSimple(model, ctx2, {
  cacheRetention: "short",
  sessionId: "notebook-08-demo",
});

const text2 = reply2.content.filter((b) => b.type === "text").map((b) => (b as { text: string }).text).join("");
console.log("💬", text2);
console.log(`\ncacheRead : ${reply2.usage.cacheRead} tokens`);
console.log(`cacheWrite: ${reply2.usage.cacheWrite} tokens`);
console.log("(0 is expected if this deployment doesn't cache — the API is the lesson here.)");

## 3. Cancellation 🛑

Real UIs need a **stop button**. Every pi-ai call accepts an `AbortSignal` via
`options.signal`. Abort it and the stream terminates promptly with an `error` event
carrying `stopReason: "aborted"` (or throws — we handle both defensively).

Here we ask for a long story, then abort after 150 ms.

In [ ]:
const ctx3: Context = {
  systemPrompt: "You are a verbose storyteller.",
  messages: [{ role: "user", content: "Tell me a long, detailed story about a lighthouse keeper.", timestamp: Date.now() }],
};

const controller = new AbortController();
setTimeout(() => controller.abort(), 150); // cancel after 150ms

let aborted3 = false;
let stopReason3 = "";
try {
  const stream3 = models.streamSimple(model, ctx3, { signal: controller.signal });
  for await (const event of stream3) {
    if (event.type === "text_delta") Deno.stdout.writeSync(new TextEncoder().encode(event.delta));
    else if (event.type === "error") { aborted3 = true; stopReason3 = event.error.stopReason; }
    else if (event.type === "done") stopReason3 = event.reason;
  }
} catch (e) {
  aborted3 = true;
  stopReason3 = "(threw) " + String(e);
}
console.log(`\n\naborted: ${aborted3}  |  stopReason: ${stopReason3 || "(completed before abort)"}`);

## 4. Timeouts, retries & introspection 🔧

Finally, the knobs that make a single call resilient and observable. All live on
`StreamOptions` (and thus `SimpleStreamOptions`):

- **`timeoutMs`** — per-request HTTP timeout.
- **`maxRetries`** — client-side retries on transient failures.
- **`maxRetryDelayMs`** — cap on how long to wait for a server-requested retry delay.
- **`onPayload(payload, model)`** — inspect (or *replace*, by returning a new value) the
  exact request body before it's sent. Return `undefined` to leave it unchanged.
- **`onResponse(response, model)`** — inspect the HTTP status/headers as the response arrives.
- **`headers` / `metadata`** — extra request headers and provider metadata.

We wire up `onPayload` and `onResponse` so you can *see* the request and response.

In [ ]:
const ctx4: Context = {
  systemPrompt: "You are concise.",
  messages: [{ role: "user", content: "Say 'ready' and nothing else.", timestamp: Date.now() }],
};

const reply4 = await models.completeSimple(model, ctx4, {
  timeoutMs: 30_000,
  maxRetries: 2,
  onPayload: (payload) => {
    console.log("→ request payload keys:", Object.keys(payload as Record<string, unknown>));
    return undefined; // leave the payload unchanged
  },
  onResponse: (res) => {
    console.log("← HTTP", res.status);
  },
});

const text4 = reply4.content.filter((b) => b.type === "text").map((b) => (b as { text: string }).text).join("");
console.log("💬", text4);
console.log("stopReason:", reply4.stopReason, "| output tokens:", reply4.usage.output);

## ✅ What just happened

1. **Cost** — put real per-token pricing on the model via `registerAzure({ modelOverrides: { cost } })`; `usage.cost` then reflects reality, and `calculateCost(model, usage)` reproduces it.
2. **Caching** — `cacheRetention` + `sessionId` opt into provider prompt caching; watch `usage.cacheRead/cacheWrite`.
3. **Cancellation** — an `AbortController.signal` cleanly stops a stream (`stopReason: "aborted"`).
4. **Robustness/introspection** — `timeoutMs`, `maxRetries`, `maxRetryDelayMs`, plus `onPayload`/`onResponse` to inspect or rewrite the wire traffic.

These options are shared by both `completeSimple` and `streamSimple`, and by every provider.

**Next:** notebook **090 — Multiple providers**, where we register more than one model and compare them side by side.